# Layer 2 pollution event features

Explore station-day-parameter features and ranked alerts. Requires gold output from:

```bash
python -m pipelines detect
```

The ensemble trains only when there are at least 20 feature rows (`MIN_EVENT_ROWS`).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path("..").resolve()
GOLD_ROOT = ROOT / "data" / "gold"

from pipelines.detection.build import read_event_alerts, read_event_features
from pipelines.detection.features import FEATURE_MODEL_COLUMNS

features = read_event_features(GOLD_ROOT)
alerts = read_event_alerts(GOLD_ROOT)
print(f"Feature rows: {len(features)}")
print(f"Alerts: {len(alerts)}")
features.head()

## 1. Feature distributions by location and parameter

In [ ]:
if features.empty:
    print("No features — run python -m pipelines detect")
else:
    features.boxplot(column="z_score", by="parameter", figsize=(8, 4))
    plt.title("Z-score by parameter")
    plt.suptitle("")
    plt.tight_layout()
    plt.show()

## 2. Time series drill-down on top alert days

In [ ]:
if alerts.empty:
    print("No alerts yet — ingest 30+ days or use synthetic fixtures in tests/test_detection.py")
else:
    top = alerts.nsmallest(5, "rank")
    display(top[["rank", "location_id", "date_local", "parameter", "alert_score", "agreement_count"]])

## 3. Regional agreement vs spatial isolation

In [ ]:
if not features.empty:
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(
        features["regional_agreement"],
        features["spatial_isolation"],
        alpha=0.7,
    )
    ax.set_xlabel("Regional agreement")
    ax.set_ylabel("Spatial isolation")
    ax.set_title("Regional event vs isolated sensor")
    plt.tight_layout()
    plt.show()

## 4. Weak label overlap with ensemble alerts (Precision@K)

In [ ]:
if alerts.empty:
    weak = features.get("weak_label")
    if weak is not None:
        print(f"Weak labels in features: {features['weak_label'].sum() if 'weak_label' in features else 'N/A'}")
    print("Precision@K requires ranked alerts — fetch more history first.")
else:
    k = min(10, len(alerts))
    top_k = alerts.nsmallest(k, "rank")
    precision = top_k["weak_label"].mean()
    pd.DataFrame(
        {
            "K": [k],
            "alerts_at_k": [len(top_k)],
            "weak_label_hits": [top_k["weak_label"].sum()],
            "precision_at_k": [precision],
        }
    )

## 5. PM2.5 vs PM10 co-movement

In [ ]:
if not features.empty:
    pm = features[features["parameter"].isin(["pm25", "pm10"])]
    pm.pivot_table(
        index=["location_id", "date_local"],
        columns="parameter",
        values="pm_co_movement",
    ).head(10)